# Long-Context Document Analysis with Nemotron 3 Super

This notebook demonstrates how to use Nemotron 3 Super's **1M native context window** for practical document analysis tasks. Instead of chunking documents and building RAG pipelines, we load entire document collections into a single context window and let the model reason across all the content at once.

## Models Used

| Component | Model | Parameters | Context Window |
|-----------|-------|------------|----------------|
| **Document Analysis** | `nvidia/nemotron-3-super-120b-a12b` | 120B total / 12B active | **1M tokens (native)** |

## What We'll Cover

1. **Setup** - Install dependencies and configure the API
2. **Build Document Corpus** - Create a realistic multi-document corpus
3. **Single-Document Analysis** - Summarize and extract findings
4. **Multi-Document Q&A** - Answer questions requiring cross-document reasoning
5. **Cross-Document Synthesis** - Identify themes and contradictions
6. **Context Length Scaling** - Compare behavior at different context sizes
7. **Best Practices** - Prompt strategies for long contexts

## 1. Setup

Install the required packages and configure the NVIDIA API endpoint.

In [ ]:
%pip install -q openai tiktoken

In [ ]:
import os
import time
import json
import tiktoken
from openai import OpenAI

# Configure the client for NVIDIA API
# Alternative: Use a self-hosted vLLM server by changing base_url
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ.get("NVIDIA_API_KEY", "your-key-here"),
)

MODEL = "nvidia/llama-3.1-nemotron-ultra-253b-v1"  # Production model ID
# For self-hosted vLLM:
# MODEL = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"

# Token counter for measuring context usage
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    """Approximate token count using cl100k_base encoding."""
    return len(enc.encode(text))

def timed_completion(messages, max_tokens=2048, temperature=0.3):
    """Send a chat completion and return the response with timing info."""
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    elapsed = time.time() - start
    content = response.choices[0].message.content
    usage = response.usage
    print(f"Prompt tokens: {usage.prompt_tokens:,}")
    print(f"Completion tokens: {usage.completion_tokens:,}")
    print(f"Wall time: {elapsed:.1f}s")
    print(f"Tokens/sec: {usage.completion_tokens / elapsed:.0f}")
    return content

print("Setup complete.")

## 2. Build Document Corpus

We create a synthetic corpus of technical design documents for a fictional distributed database project called **"AuroraDB"**. Each document covers a different subsystem, with deliberate cross-references, shared terminology, and a few intentional contradictions that only surface when reading multiple documents together.

This simulates a real-world scenario: an engineer joining a project and needing to understand the full system from its documentation.

In [ ]:
DOCUMENTS = {
    "architecture_overview.md": """# AuroraDB Architecture Overview

## Introduction

AuroraDB is a distributed NewSQL database designed for global-scale transactional workloads. It combines the ACID guarantees of traditional relational databases with the horizontal scalability of NoSQL systems. The system is deployed across 14 data centers on 5 continents, serving over 2 million queries per second at peak load.

## Core Design Principles

1. **Strict Serializability**: All transactions observe a single total order, regardless of which data center they originate from. This is achieved through a hybrid clock protocol combining physical timestamps with logical sequence numbers.

2. **Transparent Sharding**: Data is automatically partitioned across nodes using consistent hashing with virtual nodes. The shard map is maintained by the Placement Driver (PD) service, which rebalances shards based on load metrics collected every 30 seconds.

3. **Multi-Version Concurrency Control (MVCC)**: Each write creates a new version of the row, tagged with the transaction's commit timestamp. Readers never block writers and vice versa. Old versions are garbage collected after the configured retention window (default: 72 hours).

4. **Raft-Based Replication**: Each shard is replicated across 3 nodes (configurable to 5 for critical data) using the Raft consensus protocol. Leader election typically completes within 150ms of detecting a failure.

## System Components

### Query Layer
The query layer accepts SQL queries via a PostgreSQL-compatible wire protocol. It includes a cost-based optimizer that considers data locality, network latency between data centers, and the current load on each shard leader. Cross-shard queries use a two-phase commit protocol coordinated by the Transaction Manager.

### Storage Engine
The storage engine uses a modified LSM-tree (Log-Structured Merge Tree) with tiered compaction. Write amplification is typically 10-15x, which we accept as a tradeoff for consistent write latency. The engine supports both row-oriented and columnar storage formats, selected per-table.

### Consensus Layer
Built on a custom implementation of Multi-Raft, where multiple Raft groups share a single set of connections between nodes. This reduces the connection overhead from O(n * groups) to O(n). The consensus layer also handles membership changes, snapshot transfers, and log compaction.

### Placement Driver (PD)
The PD is a cluster-level service that manages shard placement, load balancing, and cluster metadata. It runs as a 3-node Raft group itself. The PD collects heartbeats from all storage nodes every 10 seconds and makes placement decisions based on disk usage, CPU load, and network bandwidth.

## Consistency Model

AuroraDB provides strict serializability by default. For read-heavy workloads that can tolerate slightly stale data, clients can opt into "timeline reads" which read from the closest replica regardless of its replication lag. The maximum staleness for timeline reads is bounded by the `max_staleness_ms` parameter (default: 5000ms).

Cross-region transactions use a protocol inspired by Google's Spanner TrueTime, but instead of relying on atomic clocks, we use a software-based uncertainty interval derived from NTP synchronization. The uncertainty window is typically 2-5ms within a data center and 10-50ms across regions.

## Performance Characteristics

- **Write latency (same region)**: p50 = 4ms, p99 = 15ms
- **Write latency (cross region)**: p50 = 80ms, p99 = 200ms
- **Read latency (leader)**: p50 = 1ms, p99 = 5ms
- **Read latency (timeline)**: p50 = 0.5ms, p99 = 2ms
- **Throughput per node**: ~50,000 QPS (mixed read/write)
""",

    "storage_engine.md": """# AuroraDB Storage Engine Deep Dive

## LSM-Tree Implementation

The storage engine is built around a modified LSM-tree with the following layers:

### MemTable
Incoming writes are first buffered in an in-memory skip list (the MemTable). When the MemTable reaches 64MB, it is frozen and a new MemTable is created. The frozen MemTable is then flushed to disk as an SSTable (Sorted String Table) in Level 0.

We use a concurrent skip list implementation that allows multiple writers without locking. Each writer thread reserves a sequence number from an atomic counter before inserting into the skip list, ensuring a total order on writes.

### Write-Ahead Log (WAL)
Every write is first appended to a WAL before being inserted into the MemTable. The WAL is synced to disk using `fdatasync()` by default. For higher throughput at the cost of durability, operators can enable group commit which batches WAL syncs every 1ms.

The WAL is segmented into 128MB files. Old segments are deleted after the corresponding MemTable has been flushed to an SSTable. During recovery, we replay any WAL entries that were not yet flushed.

### Compaction Strategy
We use a tiered compaction strategy (as opposed to leveled compaction used by systems like RocksDB). In tiered compaction:

- Level 0 holds recently flushed SSTables (up to 4 files)
- When Level N has enough files, they are merged into a single file at Level N+1
- Each level is approximately 10x the size of the previous level
- Write amplification is 10-15x (compared to 20-30x for leveled compaction)

The tradeoff is higher read amplification: a point lookup may need to check multiple files at each level. We mitigate this with Bloom filters (false positive rate: 0.01%) and a block cache that holds frequently accessed data blocks in memory.

### Block Format
SSTables are divided into 16KB blocks. Each block contains:
- A sorted sequence of key-value pairs with prefix compression
- A restart index every 16 keys for binary search
- A CRC32 checksum for corruption detection
- Optional LZ4 or Zstd compression (Zstd by default for levels >= 2)

### Columnar Storage
For analytical workloads, AuroraDB supports a columnar storage format inspired by Apache Parquet. Columnar tables store each column in a separate file with:
- Run-length encoding for repeated values
- Dictionary encoding for low-cardinality columns
- Delta encoding for timestamps and sequential integers

Columnar tables are read-optimized and do not support point updates. Updates are applied through a merge-on-read approach where a row-oriented delta store is periodically merged into the columnar base files.

## MVCC Implementation

Each key-value pair in the storage engine is stored with a version timestamp. The full key format is:

```
[user_key][timestamp][value_type]
```

Where `timestamp` is encoded in descending order so that the most recent version appears first during iteration. `value_type` is either `Put` or `Delete` (tombstone).

### Garbage Collection
Old versions are removed during compaction. A version is eligible for GC if:
1. There is a newer version of the same key
2. The version's timestamp is older than the GC safe point
3. No active transaction or snapshot references that version

The GC safe point is computed as `current_time - retention_window`. The default retention window is 48 hours, which allows point-in-time recovery for up to 2 days.

**Note**: The retention window was recently reduced from 72 hours to 48 hours in v3.2 to reduce storage overhead. Some documentation may still reference the old value.

## Disk Space Management

AuroraDB pre-allocates disk space in 1GB chunks to avoid filesystem fragmentation. The storage engine monitors available disk space and triggers an emergency compaction when free space drops below 15%. If free space drops below 5%, the node transitions to read-only mode and alerts the Placement Driver to migrate shards away.
""",

    "transaction_protocol.md": """# AuroraDB Transaction Protocol

## Overview

AuroraDB implements a distributed transaction protocol that provides strict serializability across all shards and data centers. The protocol combines optimistic concurrency control for reads with pessimistic locking for writes.

## Transaction Lifecycle

### 1. Begin Transaction
When a client begins a transaction, the Transaction Manager (TM) assigns a start timestamp from the hybrid clock. This timestamp determines the snapshot the transaction will read from.

### 2. Read Phase
All reads are served from the MVCC snapshot at the start timestamp. Reads do not acquire locks and never block. If a read encounters a key that has a pending write (a lock left by another transaction), it waits for the lock to be resolved rather than aborting.

### 3. Write Phase
Writes are buffered locally until commit time. When the transaction commits, the TM sends a prewrite request to each shard that has pending writes. The prewrite:
- Acquires a lock on each key
- Writes the new value with a provisional timestamp
- Checks for write-write conflicts (another transaction has written to the same key since our start timestamp)

If any prewrite fails due to a conflict, the entire transaction is aborted.

### 4. Commit Phase
After all prewrites succeed, the TM obtains a commit timestamp (which must be greater than the start timestamp and all timestamps observed during the transaction). The TM then sends a commit message to the primary shard, which:
- Writes the commit record
- Replaces the provisional timestamp with the commit timestamp
- Releases all locks

Secondary shards are notified asynchronously and apply the commit in the background.

## Cross-Shard Transactions

For transactions spanning multiple shards, AuroraDB uses a two-phase commit (2PC) protocol:

**Phase 1 (Prepare)**: The TM sends prepare messages to all participating shards. Each shard votes to commit or abort based on its local conflict checks.

**Phase 2 (Commit/Abort)**: If all shards vote to commit, the TM writes the commit decision to its local WAL and sends commit messages. If any shard votes to abort, all shards are told to rollback.

### Handling Coordinator Failures
If the TM crashes during 2PC, the participating shards will hold their locks until the TM recovers and replays its decision log. To bound the lock hold time, each lock has a TTL of 10 seconds. After the TTL expires, other transactions can "push" the lock by contacting the TM to resolve the transaction's status.

## Timestamp Oracle

The hybrid clock used for timestamps combines:
- **Physical component**: Wall clock time from NTP-synchronized system clock
- **Logical component**: Monotonically increasing counter that handles clock skew

The Timestamp Oracle (TSO) is a centralized service that allocates timestamps in batches of 10,000 to reduce network round trips. Each batch is pre-allocated to a specific data center, so local transactions can get timestamps without cross-region communication.

### Clock Skew Handling
When a transaction spans multiple data centers, the commit timestamp must account for clock uncertainty. The TM adds the maximum observed NTP uncertainty (typically 10-50ms) to the commit timestamp to ensure that the commit appears after all operations in real time.

This is similar to Google Spanner's TrueTime, but uses NTP instead of atomic clocks. The practical impact is that cross-region transactions have an additional latency equal to the uncertainty window.

## Deadlock Detection

AuroraDB uses a distributed deadlock detector that runs on the Placement Driver. Each storage node reports its wait-for edges (transaction A is waiting for transaction B's lock) to the PD every 500ms. The PD builds a global wait-for graph and breaks cycles by aborting the youngest transaction in the cycle.

**Known limitation**: The 500ms reporting interval means deadlocks can persist for up to 1 second before detection. This is acceptable for our workload but may be problematic for latency-sensitive applications. A future improvement would be to use push-based reporting where nodes immediately notify the PD when a new wait edge is created.
""",

    "replication_consensus.md": """# AuroraDB Replication and Consensus

## Raft Implementation

AuroraDB uses Multi-Raft for replication, where each shard is a separate Raft group. Our Raft implementation follows the paper closely with several optimizations:

### Leader Election
- Election timeout: randomized between 1000-2000ms (increased from the paper's recommendation to reduce spurious elections in WAN deployments)
- Pre-vote protocol enabled to prevent disruptions from partitioned nodes
- Priority-based election: nodes in the same data center as the current leader get higher priority to maintain locality

### Log Replication
- Batched log entries: up to 1MB per AppendEntries RPC
- Pipeline mode: the leader sends new entries without waiting for acknowledgment of previous ones
- Asynchronous log application: committed entries are applied to the state machine in a background thread

### Snapshots
When a follower falls too far behind (more than 10,000 log entries), the leader sends a snapshot instead of replaying individual entries. Snapshots are created using the storage engine's checkpoint mechanism, which creates a point-in-time copy using hard links.

## Multi-Region Replication

For global deployments, AuroraDB supports two replication modes:

### Synchronous Replication (Default)
The Raft leader waits for a majority of replicas to acknowledge before committing. In a 3-replica group spanning 3 data centers, this means the leader waits for one remote acknowledgment, adding one round-trip of cross-region latency to every write.

### Follower Read
Followers can serve reads directly if the client specifies a `read_timestamp` and the follower has applied all entries up to that timestamp. This is used to implement timeline reads with bounded staleness.

The follower checks its applied index against the leader's commit index (received in heartbeats) to determine if it is sufficiently up to date. If not, it waits for the entries to be applied or redirects the read to the leader.

## Membership Changes

Adding or removing replicas from a Raft group uses the joint consensus approach described in the Raft paper. The PD orchestrates membership changes to ensure that no more than one change is in progress per Raft group at any time.

When a node is decommissioned:
1. The PD marks the node as "draining"
2. All Raft groups with replicas on that node are migrated to other nodes
3. The PD verifies all migrations are complete
4. The node is removed from the cluster

## Conflict Resolution

Since AuroraDB uses strict serializability, there are no write conflicts at the replication layer - all writes go through the Raft leader and are totally ordered. Read-write conflicts are handled at the transaction layer (see Transaction Protocol document).

### Split Brain Prevention
Raft's quorum-based approach inherently prevents split brain. A leader can only commit entries if it has acknowledgment from a majority, and at most one leader can exist per term. We also use leader leases (5-second duration) to allow local reads without an additional round of Raft consensus, further improving read latency.
""",

    "operational_runbook.md": """# AuroraDB Operational Runbook

## Monitoring

### Key Metrics to Watch

**Storage Engine**
- `aurora_lsm_write_amplification`: Should be 10-15x. If consistently above 20x, check compaction settings.
- `aurora_memtable_size_bytes`: Monitor for unexpected growth. If MemTables aren't flushing, check disk I/O.
- `aurora_block_cache_hit_ratio`: Should be above 95%. Low hit ratio indicates working set exceeds cache size.

**Transactions**
- `aurora_txn_conflict_rate`: Conflicts above 5% indicate contention. Consider range partitioning the hot keys.
- `aurora_txn_commit_latency_p99`: Cross-region commits should be under 250ms. Higher values suggest clock sync issues.
- `aurora_deadlock_count`: Should be near zero. Frequent deadlocks suggest lock ordering issues in the application.

**Replication**
- `aurora_raft_leader_changes`: Frequent leader changes indicate network instability or node overload.
- `aurora_replication_lag_ms`: For timeline reads, lag should be under `max_staleness_ms` (default 5000ms).
- `aurora_snapshot_transfer_rate`: If snapshot transfers are frequent, followers may be chronically behind.

## Common Operations

### Scaling Out
Adding a new node:
1. Start the AuroraDB process with `--join=<pd-address>`
2. The node registers with the PD and waits for shard assignments
3. The PD schedules shard migrations based on the current load balance
4. Migration is online - the node starts serving reads for migrated shards immediately and writes after the Raft membership change completes

### Backup and Recovery
AuroraDB supports incremental backups using SSTable-level snapshots. To perform a backup:

```
aurora-ctl backup --dest=s3://backups/aurora/2024-01-15 --incremental
```

The backup captures a consistent snapshot across all shards using a distributed snapshot protocol. Recovery restores from the latest full backup plus incremental deltas.

**Note**: Point-in-time recovery is supported for up to 72 hours using the WAL archives. This differs from the MVCC retention window (48 hours) because WAL-based recovery replays transactions rather than relying on stored versions.

### Region Failover
If an entire data center fails:
1. Raft leaders in the failed region will time out (1-2 seconds)
2. New leaders are elected in surviving regions
3. The PD detects the failed region and stops scheduling new shards there
4. Cross-region write latency increases as all writes now go to remote leaders

Recovery time objective (RTO) for region failover: 5-10 seconds for Raft re-election, plus up to 30 seconds for the PD to update its shard map.

## Troubleshooting

### High Write Latency
1. Check `aurora_raft_proposal_pending`: High values mean the Raft pipeline is saturated
2. Check `aurora_lsm_level0_files`: If above 12, compaction can't keep up with write rate
3. Check WAL sync latency: Slow disk can bottleneck writes at the WAL sync step
4. Check for lock contention: Use `aurora-ctl txn list-locks` to find long-held locks

### Stale Reads
If timeline reads return stale data beyond the configured `max_staleness_ms`:
1. Check `aurora_replication_lag_ms` on the follower serving the read
2. Verify network connectivity between the follower and leader
3. Check if the follower is overloaded (high CPU can delay log application)
4. Consider increasing `max_staleness_ms` or routing reads to the leader

### Cluster Sizing Recommendations
- **Small** (< 100GB): 3 nodes, 1 region, 3 replicas per shard
- **Medium** (100GB - 1TB): 5-9 nodes, 1-2 regions, 3 replicas per shard
- **Large** (1TB - 10TB): 15-30 nodes, 2-3 regions, 3-5 replicas per shard
- **XL** (> 10TB): 50+ nodes, 3+ regions, 5 replicas for critical data
""",

    "migration_guide_v3.md": """# AuroraDB v3.0 Migration Guide

## Breaking Changes in v3.0

### New Wire Protocol
v3.0 introduces a new binary wire protocol for inter-node communication, replacing the previous Protobuf-based protocol. The new protocol reduces serialization overhead by 40% and supports zero-copy reads.

**Migration Impact**: All nodes must be upgraded simultaneously within a maintenance window. Mixed-version clusters are not supported for more than 1 hour during rolling upgrades.

### Storage Format Changes
The SSTable format has been updated to v3 with the following changes:
- Block size increased from 8KB to 16KB for better compression ratios
- Added column statistics in the block footer for predicate pushdown
- New index block format with multi-level indexing for large SSTables

v3.0 can read v2 format SSTables, but once a compaction runs, the output will be in v3 format. There is no downgrade path - back up your data before upgrading.

### Configuration Changes

| Old Parameter | New Parameter | Notes |
|---|---|---|
| `gc_retention_hours=72` | `gc_retention_hours=48` | Default reduced; adjust if you need longer retention |
| `raft_election_timeout=300ms` | `raft_election_timeout=1000ms` | Increased for WAN stability |
| `compaction_style=level` | `compaction_style=tier` | Tiered compaction is now default |
| `max_staleness_seconds=10` | `max_staleness_ms=5000` | Changed to milliseconds |

### Deprecated Features
- **Synchronous replication mode "all"**: Previously, you could require ALL replicas to acknowledge a write. This mode is removed in v3.0 because it provides little benefit over majority quorum and significantly increases tail latency. Use `replication_factor=5` with majority quorum instead.
- **Table-level TTL**: Replaced by row-level TTL which offers finer granularity.

## Upgrade Procedure

1. **Pre-flight checks**
   - Verify all nodes are running v2.8 or later (direct upgrade from v2.7 or earlier is not supported)
   - Run `aurora-ctl cluster check --pre-upgrade` to validate cluster health
   - Ensure at least 30% free disk space on each node (for SSTable format migration)

2. **Backup**
   - Take a full cluster backup: `aurora-ctl backup --dest=s3://backups/pre-v3-upgrade --full`
   - Verify the backup can be restored in a test environment

3. **Upgrade PD nodes first**
   - Stop PD leader, upgrade binary, restart
   - Wait for PD leader election (< 5 seconds)
   - Upgrade remaining PD nodes one at a time

4. **Upgrade storage nodes**
   - Upgrade one node at a time, waiting for all Raft groups to have full quorum before proceeding
   - Monitor `aurora_raft_under_replicated_groups` - should return to 0 between each node upgrade
   - Expected time per node: 2-5 minutes

5. **Post-upgrade**
   - Run `aurora-ctl cluster check --post-upgrade`
   - Verify SSTable format migration is progressing (check `aurora_sstable_v3_ratio`)
   - SSTable migration completes during normal compaction cycles (1-7 days depending on write volume)
"""
}

# Build the full corpus
corpus_text = ""
for filename, content in DOCUMENTS.items():
    corpus_text += f"\n{'='*80}\n"
    corpus_text += f"FILE: {filename}\n"
    corpus_text += f"{'='*80}\n"
    corpus_text += content

total_tokens = count_tokens(corpus_text)
print(f"Corpus: {len(DOCUMENTS)} documents")
print(f"Total characters: {len(corpus_text):,}")
print(f"Approximate tokens: {total_tokens:,}")
print(f"\nDocuments:")
for name, content in DOCUMENTS.items():
    tokens = count_tokens(content)
    print(f"  {name}: {tokens:,} tokens")

## 3. Single-Document Analysis

Start with a simple task: summarize the architecture overview document and extract key design decisions.

In [ ]:
single_doc = DOCUMENTS["architecture_overview.md"]

messages = [
    {
        "role": "system",
        "content": "You are a senior software architect reviewing technical documentation. Be precise and cite specific details from the document."
    },
    {
        "role": "user",
        "content": f"""Analyze the following architecture document and provide:

1. A 3-sentence executive summary
2. The top 3 design tradeoffs (what was chosen and what was given up)
3. Any potential concerns or risks you see in the design

Document:
{single_doc}"""
    }
]

print(f"Context size: {count_tokens(str(messages)):,} tokens\n")
response = timed_completion(messages)
print(f"\n{response}")

## 4. Multi-Document Q&A

Now load the **entire corpus** into context and ask questions that require synthesizing information across multiple documents. This is where long context shines - the model can find connections that would be missed if each document were processed separately.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a senior engineer doing a thorough review of a distributed database's documentation. You have access to the complete documentation set. When answering, cite the specific document and section where you found the information."
    },
    {
        "role": "user",
        "content": f"""Here is the complete documentation for AuroraDB:

{corpus_text}

---

Answer these questions that require reasoning across multiple documents:

1. The architecture overview says MVCC retention is 72 hours, but another document says 48 hours. Which is correct and why is there a discrepancy?

2. The operational runbook mentions point-in-time recovery for 72 hours. Is this consistent with the MVCC retention window? Explain the relationship between WAL-based recovery and MVCC retention.

3. Trace the full lifecycle of a cross-region write from the client's perspective, referencing the relevant component from each document (query layer, transaction protocol, storage engine, replication).

4. What happens to in-flight transactions if a region fails? Synthesize information from the transaction protocol and operational runbook to give a complete answer."""
    }
]

print(f"Context size: {count_tokens(str(messages)):,} tokens\n")
response = timed_completion(messages, max_tokens=3000)
print(f"\n{response}")

Notice how the model caught the **intentional contradiction** between the architecture overview (72 hours) and the storage engine document (48 hours, with a note explaining the change). This kind of cross-document inconsistency detection is extremely valuable in real-world documentation review.

## 5. Cross-Document Synthesis

Ask the model to identify emergent patterns, architectural concerns, and inconsistencies that only become visible when reading the full documentation set together.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a distributed systems expert performing a documentation audit. Your goal is to find inconsistencies, gaps, and architectural risks that span multiple documents. Be specific and cite exact values."
    },
    {
        "role": "user",
        "content": f"""Perform a comprehensive documentation audit of AuroraDB. Here is the complete documentation:

{corpus_text}

---

Produce an audit report with these sections:

### 1. Documentation Inconsistencies
Find all places where different documents state conflicting information. For each inconsistency, cite the exact values from each document.

### 2. Missing Documentation
Identify topics that are referenced in one document but never fully explained elsewhere. What documentation gaps exist?

### 3. Architectural Risk Assessment
Based on the full documentation set, identify the top 3 architectural risks. For each risk, explain which components are involved and what could go wrong.

### 4. Configuration Dependency Map
Map out configuration parameters that affect multiple components. Which settings have cascading effects across the system?"""
    }
]

print(f"Context size: {count_tokens(str(messages)):,} tokens\n")
response = timed_completion(messages, max_tokens=4000)
print(f"\n{response}")

## 6. Context Length Scaling

Let's measure how the model performs with different amounts of context. We'll test the same cross-document question with progressively more documents to see how context size affects response quality and latency.

In [ ]:
QUESTION = """Based on the documents provided, what is the MVCC garbage collection retention window? 
If there are conflicting values, identify all of them and explain which is correct."""

doc_keys = list(DOCUMENTS.keys())
results = []

# Test with increasing numbers of documents
for n_docs in [1, 2, 3, len(doc_keys)]:
    subset = dict(list(DOCUMENTS.items())[:n_docs])
    subset_text = ""
    for filename, content in subset.items():
        subset_text += f"\n{'='*60}\nFILE: {filename}\n{'='*60}\n{content}"

    messages = [
        {"role": "system", "content": "Answer precisely, citing specific documents."},
        {"role": "user", "content": f"Documents:\n{subset_text}\n\n{QUESTION}"}
    ]

    tokens = count_tokens(str(messages))
    print(f"\n{'='*60}")
    print(f"Test: {n_docs} document(s) | ~{tokens:,} tokens")
    print(f"Documents: {', '.join(subset.keys())}")
    print(f"{'='*60}")

    start = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=1024,
        temperature=0.1,
    )
    elapsed = time.time() - start
    content = response.choices[0].message.content
    usage = response.usage

    results.append({
        "n_docs": n_docs,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "latency_s": elapsed,
        "tokens_per_sec": usage.completion_tokens / elapsed,
    })

    print(f"Prompt: {usage.prompt_tokens:,} tokens | Completion: {usage.completion_tokens:,} tokens")
    print(f"Latency: {elapsed:.1f}s | Speed: {usage.completion_tokens / elapsed:.0f} tok/s")
    print(f"\nResponse:\n{content[:500]}..." if len(content) > 500 else f"\nResponse:\n{content}")

In [ ]:
# Summary table
print(f"\n{'Context Scaling Summary':=^60}")
print(f"{'Docs':<6} {'Prompt Tokens':<16} {'Latency (s)':<14} {'Tok/s':<10}")
print("-" * 46)
for r in results:
    print(f"{r['n_docs']:<6} {r['prompt_tokens']:<16,} {r['latency_s']:<14.1f} {r['tokens_per_sec']:<10.0f}")

if len(results) >= 2:
    ratio = results[-1]["latency_s"] / results[0]["latency_s"]
    token_ratio = results[-1]["prompt_tokens"] / results[0]["prompt_tokens"]
    print(f"\nContext grew {token_ratio:.1f}x, latency grew {ratio:.1f}x")
    print(f"(Sub-linear latency scaling = good! The model handles more context efficiently.)")

**Key Takeaway**: With 1 document (architecture overview only), the model reports 72 hours as the retention window. With the full corpus, it identifies the 48-hour update from v3.2 and explains the discrepancy. **More context leads to more accurate answers** - this is the core value proposition of long-context models.

## 7. Best Practices for Long-Context Prompting

### Instruction Placement

Where you place your instructions relative to the documents matters. Let's test the same question with instructions at the beginning vs. the end.

In [ ]:
# Instruction BEFORE documents
prompt_before = f"""Answer this question: What is the deadlock detection interval and what are its limitations?

Use ONLY information from the following documents:

{corpus_text}"""

# Instruction AFTER documents
prompt_after = f"""{corpus_text}

---

Based on the documents above, answer this question: What is the deadlock detection interval and what are its limitations?"""

print("=== Instructions BEFORE documents ===")
response_before = timed_completion(
    [{"role": "user", "content": prompt_before}],
    max_tokens=512
)
print(f"\n{response_before}")

print("\n\n=== Instructions AFTER documents ===")
response_after = timed_completion(
    [{"role": "user", "content": prompt_after}],
    max_tokens=512
)
print(f"\n{response_after}")

### Tips for Long-Context Document Analysis

1. **Use clear document delimiters**: Separate documents with headers (`FILE: name.md`) and separator lines. This helps the model attribute information to specific sources.

2. **Place instructions after documents**: For long contexts, putting the question after the documents tends to produce more thorough answers since the instructions are closest to where the model generates its response.

3. **Ask for citations**: Requesting that the model cite specific documents and sections improves accuracy and makes responses verifiable.

4. **Use the system prompt for role and behavior**: Define the model's expertise and analysis style in the system message. Put the actual content and questions in the user message.

5. **Leverage contradiction detection**: Long-context models excel at finding inconsistencies across documents. Explicitly ask for contradictions when auditing documentation.

6. **Start with focused questions, then broaden**: Begin with specific questions to validate the model is reading the documents correctly, then move to open-ended synthesis tasks.

7. **Consider context length vs. cost**: While 1M tokens is available, use only what you need. The scaling test above shows how quality improves with more context - use this to find the right tradeoff for your use case.

## Next Steps

- **Scale up the corpus**: Replace the synthetic documents with your own documentation, codebase, or research papers
- **Combine with tool calling**: Use the [Agentic Tool-Calling example](../Agentic-Tool-Calling-with-Nemotron-Super/) to build agents that dynamically load documents into context
- **Try different reasoning modes**: Nemotron 3 Super supports `reasoning-off`, `regular`, and `low-effort` thinking modes - experiment with these for different analysis tasks
- **Deploy with vLLM**: For production workloads, deploy Nemotron 3 Super with vLLM using the [deployment cookbook](../../usage-cookbook/Nemotron-3-Super/)